# 10 Point clouds for Potree

Turns the TNG300-1 subhalo samples behind the training inputs into 3D point clouds for [Potree](https://github.com/potree/potree), one per spacing λ = 3, 5, 7, 10 h⁻¹ Mpc. Each subhalo is colored by the T-web class of the voxel it sits in.

1. Downloads `SubhaloPos`, `SubhaloMass` and `SubhaloVel` of the TNG300-1 snapshot-99 group catalog from the TNG API and saves them to Drive (once).
2. Checks that the N most massive subhalos reproduce the gridded `tng300-3/subhalos/L*` fields the models are trained on.
3. Writes one LAS file per spacing: classification = T-web class (0 void, 1 wall, 2 filament, 3 halo), plus log₁₀ of the subhalo mass.
4. Builds PotreeConverter, converts the LAS files to octrees, and writes a viewer page.

Needs Colab secrets `GITHUB_TOKEN` and `TNG_API_KEY` (from your profile page at tng-project.org).

Runtime: CPU with **High-RAM**; no GPU. About 15 minutes, most of it the catalog download and building PotreeConverter.

In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive')
token = userdata.get('GITHUB_TOKEN')
!rm -rf /content/deepvoid
!git clone -q "https://{token}@github.com/skumagai3/deepvoid.git" /content/deepvoid
!git -C /content/deepvoid remote set-url origin https://github.com/skumagai3/deepvoid.git
!pip -q install --force-reinstall --no-deps /content/deepvoid
!pip -q install "laspy[lazrs]"
del token
!git -C /content/deepvoid log --oneline -1

In [ ]:
from pathlib import Path
import numpy as np
from deepvoid import Catalog
from deepvoid.labels import CLASS_NAMES

catalog = Catalog()
SIM = 'TNG300-1'
SPACINGS = (3, 5, 7, 10)
subhalo_file = catalog.root / 'TNG' / f'{SIM}_subhalos_snap99.npz'
out = Path('/content/drive/MyDrive/deepvoid_viz/subhalos_tweb')

## 1. Subhalo catalog

About 14.5 million subhalos for TNG300-1. Skipped if the file is already on Drive.

In [ ]:
from deepvoid.sources.tng import fetch_subhalos, read_subhalos

if not subhalo_file.exists():
    fetch_subhalos(subhalo_file, api_key=userdata.get('TNG_API_KEY'), sim=SIM, tmp_dir='/content')
subhalos = read_subhalos(subhalo_file)
print(f'{len(subhalos):,} subhalos from {subhalos.meta["sim"]}, saved at {subhalo_file}')

## 2. Check the selection against the training inputs

For each spacing, keep the N = (205 / λ)³ most massive subhalos, grid them with mass-weighted CIC on the 512³ grid, and correlate with the field the models were trained on. A correlation of 1 means this is the same sample, gridded the same way. Both grid conventions are tried: grid points at integer multiples of the cell size (the package convention) and at cell centers. The better one decides which voxel each subhalo is assigned to below.

If the correlations are well below 1, the training fields came from a different sample: try `SIM = 'TNG300-1-Dark'` in the cell above and rerun from section 1.

In [ ]:
import pandas as pd
from deepvoid.observe import select_by_spacing
from deepvoid.pointcloud import compare_with_field

samples, rows = {}, {}
for lam in SPACINGS:
    samples[lam] = select_by_spacing(subhalos, lam)
    field = catalog.load(f'tng300-3/subhalos/L{lam}').data
    rows[f'L{lam}'] = compare_with_field(samples[lam], field)
    del field
check = pd.DataFrame(rows).T
print(check.round(4).to_string())

centered = check['correlation_centered'].mean() > check['correlation_integer'].mean()
print('\nvoxel assignment:', 'cell-centered grid' if centered else 'integer grid (package convention)')

## 3. LAS files colored by T-web class

The T-web mask is computed from the TNG300-3-Dark matter field, the same mask the models are trained against. TNG300-1 and TNG300-3 share initial conditions, so their large-scale structure lines up. The table compares the class fractions of the subhalos with those of all voxels: subhalos sit mostly in filaments and halos, so voids are rare among them.

In [ ]:
from deepvoid.pointcloud import values_at, write_las

mask = catalog.load('tng300-3/mask').data
fractions = {'all voxels': np.bincount(mask.ravel(), minlength=4) / mask.size}
for lam, t in samples.items():
    cls = values_at(mask, t.pos, t.box_size, centered=centered)
    write_las(out / 'las' / f'subhalos_L{lam}.las', t.pos, cls,
              extra={'log_mass': np.log10(t.mass).astype(np.float32)})
    fractions[f'L{lam} ({len(t):,} subhalos)'] = np.bincount(cls, minlength=4) / len(cls)
print('Fraction in each T-web class')
print(pd.DataFrame(fractions, index=CLASS_NAMES).T.round(3).to_string())

## 4. Potree octrees and viewer page

Builds PotreeConverter 2.1.1 from source (a few minutes), converts each LAS file, and writes `index.html`. The page shows λ = 3 first; the others are listed under *Scene* in the sidebar.

In [ ]:
!apt-get -qq install -y libtbb-dev > /dev/null
!rm -rf /content/PotreeConverter
!git clone -q --depth 1 --branch 2.1.1 https://github.com/potree/PotreeConverter.git /content/PotreeConverter
!cmake -S /content/PotreeConverter -B /content/PotreeConverter/build -DCMAKE_BUILD_TYPE=Release > /dev/null
!make -s -C /content/PotreeConverter/build -j8 2>&1 | tail -3
converter = '/content/PotreeConverter/build/PotreeConverter'
!{converter} --help | head -5

In [ ]:
import shutil, subprocess
from deepvoid.pointcloud import write_viewer

page = Path('/content/potree_page')
shutil.rmtree(page, ignore_errors=True)

# A generated page supplies the Potree libraries; the cloud it converts is discarded.
subprocess.run([converter, str(out / 'las' / f'subhalos_L{SPACINGS[-1]}.las'),
                '-o', str(page), '--generate-page', 'template'], check=True)
template = next(p for p in page.rglob('*.html') if 'Potree.Viewer' in p.read_text())
for meta in page.rglob('metadata.json'):
    shutil.rmtree(meta.parent)

clouds = {}
for lam in SPACINGS:
    target = template.parent / 'pointclouds' / f'L{lam}'
    subprocess.run([converter, str(out / 'las' / f'subhalos_L{lam}.las'), '-o', str(target)],
                   check=True, capture_output=True)
    clouds[f'λ = {lam} h⁻¹ Mpc'] = target / 'metadata.json'
write_viewer(template, template.parent / 'index.html', clouds,
             title='TNG300-1 subhalos by T-web class')
template.unlink()

shutil.rmtree(out / 'potree', ignore_errors=True)
shutil.copytree(template.parent, out / 'potree')
print('viewer written to', out / 'potree' / 'index.html')

## Viewing

Browsers won't load Potree pages straight from disk, so serve the folder locally:

1. Download `MyDrive/deepvoid_viz/subhalos_tweb/potree` to your computer.
2. In a terminal, in that folder: `python3 -m http.server 8000`
3. Open http://localhost:8000/index.html

In the sidebar:
- **Scene** switches the spacings on and off.
- **Filters → Classification** hides or shows each class.
- A cloud's **Attributes** setting switches the coloring from classification to `log_mass`.
- **Tools → Clip** cuts out slabs and boxes.